In [ ]:
import geopandas as gpd
import pandas as pd 
import numpy as np

In [2]:
rc_lookups = pd.read_csv("output/rc_lookups.csv")
rc_ldc =pd.read_csv("output/retail_centers_with_ldc_retailN.csv")

In [4]:
rc_lookups

,RC_ID,geometry,LAD24NM,CountyName,RegionName,gsscode
0,6025,MULTIPOLYGON (((440445.7388613202 367018.09139...,North East Derbyshire,Derbyshire,East Midlands,E63008766
1,1193,MULTIPOLYGON (((439116.7765939621 363407.88751...,North East Derbyshire,Derbyshire,East Midlands,E63008809
2,6644,MULTIPOLYGON (((194309.88097327633 76200.71289...,Cornwall,Cornwall,South West,E63013978
3,6026,MULTIPOLYGON (((439008.4176877096 364010.85388...,North East Derbyshire,Derbyshire,East Midlands,E63008809
4,7407,MULTIPOLYGON (((438294.8993448653 371476.61951...,Chesterfield,Derbyshire,East Midlands,E63008717
...,...,...,...,...,...,...
9617,2844,MULTIPOLYGON (((111614.72348488995 512113.4056...,"Armagh City, Banbridge and Craigavon",Armagh,NI,NI
9618,4025,MULTIPOLYGON (((578163.6762067255 233984.52018...,Braintree,Essex,East of England,E63011054
9619,2676,MULTIPOLYGON (((401222.3074132583 90489.386812...,"Bournemouth, Christchurch and Poole",Dorset,South West,E63013860
9620,3757,MULTIPOLYGON (((255356.56308415128 666668.6817...,Glasgow City,City of glasgow,Scotland,S45001180


In [6]:
rc = rc_ldc.merge(
     rc_lookups.drop(columns=['geometry']),
    on='RC_ID',
    how='left'
)
rc

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,geometry,LDC_retailN,LAD24NM,CountyName,RegionName,gsscode
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((440445.7388613202 367018.09139...,2.0,North East Derbyshire,Derbyshire,East Midlands,E63008766
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,0,0,MULTIPOLYGON (((439116.7765939621 363407.88751...,131.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,0,0,MULTIPOLYGON (((194309.88097327633 76200.71289...,10.0,Cornwall,Cornwall,South West,E63013978
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((439008.4176877096 364010.85388...,17.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,0,0,MULTIPOLYGON (((438294.8993448653 371476.61951...,8.0,Chesterfield,Derbyshire,East Midlands,E63008717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9617,2844,46,16,32.0,13.0,1.0,0.0,['Convenience Store' 'Discount Store' 'Restaur...,retail_park,"Armagh City, Banbridge and Craigavon",0,0,MULTIPOLYGON (((111614.72348488995 512113.4056...,59.0,"Armagh City, Banbridge and Craigavon",Armagh,NI,NI
9618,4025,29,22,17.0,8.0,2.0,2.0,['Drugstore' 'Pet Service' 'Convenience Store'...,retail_park,Braintree,0,0,MULTIPOLYGON (((578163.6762067255 233984.52018...,14.0,Braintree,Essex,East of England,E63011054
9619,2676,50,11,34.0,14.0,2.0,0.0,['Textiles Store' 'Electronics Store' 'Sportin...,retail_park,"Bournemouth, Christchurch and Poole",0,0,MULTIPOLYGON (((401222.3074132583 90489.386812...,46.0,"Bournemouth, Christchurch and Poole",Dorset,South West,E63013860
9620,3757,32,22,26.0,6.0,0.0,0.0,['Sporting Goods Retail' 'Convenience Store' '...,retail_park,Glasgow City,0,0,MULTIPOLYGON (((255356.56308415128 666668.6817...,50.0,Glasgow City,City of glasgow,Scotland,S45001180


In [9]:
rc[rc['gsscode'].isna()]['centre_type'].value_counts()  

centre_type
centre         83
retail_park     8
Name: count, dtype: int64

In [16]:
rc.columns

Index(['RC_ID', 'Total_POI_count', 'H3_cells', 'Retail', 'Dining_and_Drinking',
       'Leisure_Travel', 'Community_Health', 'category_names', 'centre_type',
       'LocalAuthorityNM_1', 'LocalAuthorityNM_2', 'LocalAuthorityNM_3',
       'geometry', 'LDC_retailN', 'LAD24NM', 'CountyName', 'RegionName',
       'gsscode'],
      dtype='object')

In [43]:
def apply_hierarchical_classification(df: pd.DataFrame, retail_park_rc_ids=None) -> pd.DataFrame:
    df = df.copy()
    retail_park_rc_ids = set() if retail_park_rc_ids is None else set(retail_park_rc_ids)

    # 1. HANDLE NA VALUES PROPERLY
    # Instead of filling with "NA" before ranking, we rank first 
    # and then handle the NAs to match the R script logic.
    rank_specs = {
        "RGN_rank": "RegionName",
        "CTY_rank": "CountyName",
        "LAD_rank": "LAD24NM",
        "BUA_rank": "gsscode",
    }
    
    for rank_col, group_col in rank_specs.items():
        # Only rank where the group name is NOT null
        mask = df[group_col].notna()
        df.loc[mask, rank_col] = df[mask].groupby(group_col)["LDC_retailN"].rank(
            method="first", ascending=False
        )
        # If it was NA, give it a massive rank so it's never Rank 1
        df[rank_col] = df[rank_col].fillna(999999)

    df["tr_retailN"] = df["LDC_retailN"]
    df["Classification"] = pd.Series(pd.NA, index=df.index)

    def assign(label: str, mask: pd.Series) -> None:
        df.loc[df["Classification"].isna() & mask, "Classification"] = label

    # TIER 1: Regional (With Manual Overrides)
    assign("Regional Centre", (df["RGN_rank"] == 1) | df["RC_ID"].isin({"RC_EW_2751", "RC_SC_280"}))

    # TIER 2: Major Town
    assign("Major Town Centre", (df["CTY_rank"] == 1) & (df["RGN_rank"] != 1) & (df["LDC_retailN"] >= 350))

    # TIER 3: Town Centre
    assign("Town Centre", 
        ((df["CTY_rank"] == 1) & (df["RGN_rank"] != 1)) | 
        ((df["LAD_rank"] == 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1)) | 
        ((df["LDC_retailN"] >= 250) & (df["LDC_retailN"] < 350))
    )

    # TIER 4: Market Town
    assign("Market Town", 
        (df["BUA_rank"] == 1) & (df["LAD_rank"] != 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1) & 
        (df["tr_retailN"] >= 150) & (df["tr_retailN"] < 250)
    )

    # TIER 5: District Centre
    assign("District Centre", 
        (df["BUA_rank"] != 1) & (df["LAD_rank"] != 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1) & 
        (df["tr_retailN"] >= 150)
    )

    # TIER 6: Local Centre
    assign("Local Centre", df["tr_retailN"] < 150)

    # REFINEMENT: Local -> Small Local
    df.loc[(df["Classification"] == "Local Centre") & (df["tr_retailN"] < 100), "Classification"] = "Small Local Centre"

    # FALLBACKS
    df.loc[df["Classification"].isna() & df["LDC_retailN"].notna(), "Classification"] = "Market Town"
    df["Classification"] = df["Classification"].fillna("Small Local Centre")

    # RETAIL PARK ADJUSTMENT
    # Refined to include District Centres if they are actually Retail Parks
    retail_park_mask = (
        (df["centre_type"] == "retail_park") | 
        df["RC_ID"].isin(retail_park_rc_ids)
    ) & (df["Classification"].isin(["Small Local Centre", "Local Centre", "District Centre"]))
    
    df.loc[retail_park_mask, "Classification"] = "Retail Park"
    df.loc[(df["Classification"] == "Retail Park") & (df["tr_retailN"] < 50), "Classification"] = "Small Retail Park"
    df.loc[(df["Classification"] == "Retail Park") & (df["tr_retailN"] >= 50), "Classification"] = "Large Retail Park"

    return df.sort_values(["RegionName", "RC_ID"]).reset_index(drop=True)

In [49]:
df= apply_hierarchical_classification(rc)
df['Classification'].value_counts()

Classification
Small Local Centre    7083
Small Retail Park     1079
Town Centre            438
Local Centre           392
Market Town            238
District Centre        236
Large Retail Park       90
Major Town Centre       54
Regional Centre         12
Name: count, dtype: int64

In [50]:
df

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LAD24NM,CountyName,RegionName,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
0,13,1361,337,756.0,454.0,119.0,32.0,['Tourist Information and Service' 'Electronic...,centre,Derby,...,Derby,Derbyshire,East Midlands,E63009183,1.0,1.0,1.0,1.0,1138.0,Regional Centre
1,39,1011,506,589.0,299.0,94.0,29.0,['Electronics Store' 'Gastropub' 'Tourist Info...,centre,West Northamptonshire,...,West Northamptonshire,Northamptonshire,East Midlands,E63010463,2.0,1.0,1.0,1.0,923.0,Major Town Centre
2,78,808,139,422.0,276.0,98.0,12.0,"['Electronics Store' ""Doctor's Office"" 'Touris...",centre,Nottingham,...,Nottingham,Nottinghamshire,East Midlands,E63009088,5.0,1.0,1.0,1.0,612.0,Major Town Centre
3,82,798,120,497.0,234.0,59.0,8.0,['Tourist Information and Service' 'Gastropub'...,centre,Leicester,...,Leicester,Leicestershire,East Midlands,E63009743,3.0,1.0,1.0,1.0,672.0,Major Town Centre
4,93,765,120,447.0,241.0,60.0,17.0,"['Electronics Store' 'Gastropub' ""Doctor's Off...",centre,Lincoln,...,Lincoln,Lincolnshire,East Midlands,E63008729,6.0,1.0,1.0,1.0,586.0,Major Town Centre
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9617,9590,2,19,2.0,0.0,0.0,0.0,['Furniture and Home Store'],retail_park,Sheffield,...,Sheffield,South yorkshire,Yorkshire and The Humber,E63008489,755.0,210.0,105.0,100.0,4.0,Small Retail Park
9618,9597,1,7,1.0,0.0,0.0,0.0,['Outdoor Supply Store'],retail_park,North Lincolnshire,...,North Lincolnshire,Lincolnshire,Yorkshire and The Humber,E63008235,765.0,104.0,14.0,8.0,3.0,Small Retail Park
9619,9600,1,35,1.0,0.0,0.0,0.0,['Hardware Store'],retail_park,Doncaster,...,Doncaster,South yorkshire,Yorkshire and The Humber,E63008314,766.0,213.0,44.0,14.0,3.0,Small Retail Park
9620,9603,1,26,1.0,0.0,0.0,0.0,['Furniture and Home Store'],retail_park,Bradford,...,Bradford,West yorkshire,Yorkshire and The Humber,E63007920,640.0,288.0,65.0,46.0,8.0,Small Retail Park


In [ ]:
def apply_shopping_centre_overrides(df, outlets_manual_list=None):
    """
    Replicates the 'MANUAL EDITS TO CLASSIFICATIONS/ SHOPPING OUTLETS' 
    section of the R code.
    """
    
    # 1. THE HARD-CODED LIST (from the R script provided)
    # The R code classifies these based on a specific manual table
    regional_shopping_names = [
        "Westfield London", "Metrocentre", "Trafford Centre", 
        "Westfield Stratford City", "Bluewater", "intu Merry Hill", 
        "Lakeside Shopping Centre", "Meadowhall", "East Kilbride Shopping Centre", 
        "Silverburn Centre", "intu Braehead", "White Rose Centre"
    ]
    
    # Large Outlets / Shopping Centres
    large_outlet_names = [
        "Ashford Designer Outlet", "Bicester Outlet Village", "Cheshire Oaks Designer Outlet",
        "Gloucester Quays Outlet", "York Designer Outlet", "Cribbs Causeway",
        "Livingston Designer Outlet", "London Designer Outlet Wembley"
        # ... you can add the full list from the R script here ...
    ]

    # 2. IDENTIFY BY NAME OR ID
    # In the R code, they intersect spatial points. In Python, assuming you have 
    # a name column or the specific RC_IDs from the R script:
    
    # Rule A: Large Shopping Centres (The R code calls these Regional Shopping Centres)
    # They are generally Out of Town and > 50 units (or manually identified)
    is_large_shopping = (
        (df['RC_Name'].str.contains('|'.join(regional_shopping_names), na=False)) |
        (df['RC_Name'].str.contains('|'.join(large_outlet_names), na=False)) |
        (df['RC_ID'].isin(["RC_EW_1312", "RC_EW_1529", "RC_EW_762", "RC_EW_3751", "RC_EW_955"]))
    )
    
    # Apply Large Classification
    df.loc[is_large_shopping, 'Classification'] = "Large Shopping Centre"

    # Rule B: Small Shopping Centres 
    # Defined as Out of Town outlets with under 50 units
    # This acts as a catch-all for shopping-type centres that weren't in the 'Large' list
    is_small_shopping = (
        (df['Classification'].isna()) & 
        (df['centre_type'] == 'shopping_centre') & 
        (df['LDC_retailN'] < 50)
    )
    
    df.loc[is_small_shopping, 'Classification'] = "Small Shopping Centre"

    return df